In [17]:
!pip install python-dotenv
!pip install crewai
!pip install langchain-openai
from dotenv import load_dotenv
from openai import OpenAI
from crewai import Agent, Task, Crew
from langchain_openai import ChatOpenAI
import os


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
load_dotenv()
client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL")
)

In [19]:
# === 1. Researcher Agent ===
researcher = Agent(
    role="Researcher",
    goal="Collect scientifically accurate raw facts about attention in neural networks.",
    backstory="Expert who extracts precise factual data without interpretation.",
    llm=client,
    max_tokens=800,
    temperature=0.2,
    verbose=True,
    allow_delegation=False,
    memory=False,
    system_prompt="""
You are Researcher Agent. Provide only precise, scientifically correct raw facts.
Do NOT write full sentences — only factual bullet points.

Collect facts for EXACTLY these categories:
1) Definition of attention mechanism  
2) Problem it solves  
3) Step-by-step mechanism  
4) Types of attention: self-attention, cross-attention, multi-head attention  
5) Why transformers rely on attention  

Requirements:
- No history  
- No applications  
- No examples  
- No interpretations  
- No explanations  
- No synonyms for technical terms  

Use ONLY the standard technical terminology used in deep learning literature.
"""

)

# === 2. Analyst Agent ===
analyst = Agent(
    role="Analyst",
    goal="Convert raw facts into a clean hierarchical outline.",
    backstory="Expert in structuring technical information.",
    llm=client,
    max_tokens=900,
    temperature=0.2,
    verbose=True,
    allow_delegation=False,
    memory=True,
    system_prompt="""
You are Analyst Agent. Transform raw research facts into a clean, hierarchical outline.

Mandatory structure:
1. Introduction  
2. Problem attention solves  
3. Step-by-step mechanism  
4. Types of attention  
5. Role of attention in transformers  
6. Summary  

Rules:
- Use ONLY bullet points  
- Do NOT add new facts  
- Do NOT rephrase technical terms  
- Do NOT write full sentences  
- Ensure perfect logical flow  
- Ensure terminology: "самоувага", "перехресна увага", "багатоголова увага"

Your outline must be minimalistic, clean, and academically structured.
"""

)

# === 3. Writer Agent ===
writer = Agent(
    role="Writer",
    goal="Write an academic explanation based strictly on the outline.",
    backstory="Technical writer who produces clean, coherent text.",
    llm=client,
    max_tokens=1200,
    temperature=0.2,
    verbose=True,
    allow_delegation=False,
    memory=True,
    system_prompt="""
You are Writer Agent. Your task is to write a polished academic explanation in Ukrainian
STRICTLY based on the Analyst’s outline.

STYLE REQUIREMENTS:
- Academic, concise, technically precise  
- No water, no repetitions, no tautology  
- No vague or filler phrases  
- Smooth transitions between sections  
- Clear structure  
- No unnecessary adjectives  
- No metaphors  
- No invented terms  

TERMINOLOGY RULES:
Use ONLY these technical translations:
- self-attention → самоувага  
- cross-attention → перехресна увага  
- multi-head attention → багатоголова увага  

WRITING RULES:
- Expand only what is in the outline  
- Do NOT add new facts  
- Do NOT contradict the outline  
- Do NOT generalize beyond the given material  
- Sentences must be grammatically perfect  
- Text must sound like a real academic Ukrainian explanation from a university lab report  

Formatting:
- Write structured paragraphs  
- Do not number sentences  
"""

)

In [20]:
research_task = Task(
    description=(
        "Collect raw factual information ONLY for the following categories: "
        "1) Definition of attention mechanism; "
        "2) Problem attention solves; "
        "3) Step-by-step mechanism; "
        "4) Types of attention (self, cross, multi-head); "
        "5) Why transformers rely on attention. "
        "Do NOT include history, applications or examples. "
        "Do NOT analyze — only provide raw facts."
    ),
    agent=researcher,
    expected_output="Raw factual data grouped by the 5 categories.",
    name="task1",
    save_output_to_memory=True
)

analysis_task = Task(
    description=(
        "Transform the research facts into a structured outline "
        "with EXACTLY the following sections: "
        "1. Introduction; "
        "2. Problem attention solves; "
        "3. Step-by-step mechanism; "
        "4. Types of attention; "
        "5. Role of attention in transformers; "
        "6. Summary. "
        "Use bullet points only. Do NOT write full sentences."
    ),
    agent=analyst,
    expected_output="Bullet-point outline with exactly 6 sections.",
    name="task2",
    memory_keys=["task1"],
    save_output_to_memory=True
)

writing_task = Task(
    description=(
        "Write a polished academic explanation in Ukrainian, "
        "strictly following the outline provided by the Analyst. "
        "Do NOT add new facts — expand only what is in the outline."
    ),
    agent=writer,
    expected_output="Full academic explanation in Ukrainian.",
    name="task3",
    memory_keys=["task2"]
)

In [21]:
crew = Crew(
    agents=[researcher, analyst, writer],
    tasks=[research_task, analysis_task, writing_task],
    verbose=True
)

result = crew.kickoff()

print("\n=== FINAL RESULT ===\n")
print(result)

def crewoutput_to_str(obj):
    for attr in ("text", "output", "final", "final_output", "result", "content"):
        if hasattr(obj, attr):
            val = getattr(obj, attr)
            if isinstance(val, str):
                return val
            try:
                return str(val)
            except Exception:
                continue
    if hasattr(obj, "dict"):
        import json
        try:
            return json.dumps(obj.dict(), ensure_ascii=False, indent=2)
        except Exception:
            pass
    return str(obj)

text = crewoutput_to_str(result)

with open("final_report.txt", "w", encoding="utf-8") as f:
    f.write(text)
    
print("=== WORKFLOW FINISHED ===")


╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 0bef1f60-a6a1-4ecb-8049-9108966b84a9                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Researcher                                                                                              │
│                                                                                                                 │
│  Task: Collect raw factual information ONLY for the following categories: 1) Definition of attention            │
│  mechanism; 2) Problem attention solves; 3) Step-by-step mechanism; 4) Types of attention (self, cross,         │
│  multi-head); 5) Why transformers rely on attention. Do NOT include history, applications or examples. Do NOT   │
│  analyze — only provide raw facts.                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

c:\A MY FILES\PROGRAMMING\Polytecnic\Neural-network-technologies-and-smart-systems\4\venv\Lib\site-packages\rich\live.py:256: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Researcher                                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Category 1: Definition of Attention Mechanism**                                                              │
│                                                                                                                 │
│  - The attention mechanism is a neural network component that focuses on specific input elements or aspects.    │
│  - It weights and combines inputs to produce a weighted sum, allowing the model to focus on relevant            │
│  information.                                                                                                   │
│  - The attention mechanism is based on calculating attention weights from the input and context.                │
│                                                                                                                 │
│  **Category 2: Problem Attention Solves**                                                                       │
│                                                                                                                 │
│  - The attention mechanism solves the problem of long-range dependencies in sequential data by selectively      │
│  focusing on relevant parts.                                                                                    │
│  - It helps models understand complex relationships between input elements.                                     │
│  - Attention addresses the issue of vanishing gradients during backpropagation through time.                    │
│                                                                                                                 │
│  **Category 3: Step-by-step Mechanism**                                                                         │
│                                                                                                                 │
│  1. Input (Query, Key, Value): The attention mechanism takes three inputs - query, key, and value vectors.      │
│  2. Calculate Attention Weights: It computes the dot product between query and key vectors to calculate         │
│  attention weights.                                                                                             │
│  3. Normalize Attention Weights: The calculated weights are normalized using a softmax function.                │
│  4. Compute Context Vector: The weighted sum of value vectors is computed based on the normalized attention     │
│  weights.                                                                                                       │
│  5. Output (Context Vector): The final output is the context vector, which represents the weighted combination  │
│  of input elements.                                                                                             │
│                                                                                                                 │
│  **Category 4: Types of Attention**                                                                             │
│                                                                                                                 │
│  - **Self-Attention:** A self-attention mechanism where the query, key, and value are derived from the same     │
│  sequence or data.                                                                                              │
│  - **Cross-Attention:** A cross-attention mechanism whe

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: task1                                                                                                    │
│  Agent: Researcher                                                                                              │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: task3                                                                                                    │
│  Agent: Writer                                                                                                  │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


=== FINAL RESULT ===


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 0bef1f60-a6a1-4ecb-8049-9108966b84a9                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: Інформація про увагу                                                                             │
│                                                                                                                 │
│  1. Введення                                                                                                    │
│     • Механізм уваги є компонентом мережі нейронів.                                                             │
│     • Він фокусується на певних елементах або аспектах вхідної інформації.                                      │
│     • Вважає та поєднує вхідні дані для виробництва вагової суми.                                               │
│                                                                                                                 │
│  2. Задача, яку розв'язує механізм уваги                                                                        │
│     • Проблема довгострокових залежностей у послідовній даних.                                                  │
│     • Поніння складних зв’язків між елементами вхідної інформації.                                              │
│     • Ванішання градієнтів під час зворотньої передачі через час.                                               │
│                                                                                                                 │
│  3. Шагова механізм                                                                                             │
│      1. Вихід (Питання, Ключ, Значення)                                                                         │
│         • Механізм уваги приймає три вхідних дані – запит, ключ і значення вектори.                             │
│      2. Рахунок вагових коефіцієнтів                                                                            │
│          • Він розрахований добуток між запит та ключ векторами для розрахунку вагових коефіцієнтів.            │
│      3. Нормалізація вагових коефіцієнтів                                                                       │
│          • Розраховані вагові коефіцієнти нормалізуються за допомогою функції softmax.                          │
│      4. Обрахунок контекстного вектору                                                                          │
│          • Вагова сума значення векторів розрахується на основі нормалізованих вагових коефіцієнтів.            │
│      5. Вихід (Контекстний вектор)                                                                              │
│         • Останній вихід – це контекстовий вектор, який представляє вагову комбінацію елементів вхідної         │
│  інформації.                                                                                                    │
│                                                                                                                 │
│  4. Типи уваги                                                                                                  │
│     • **Самовідчування:** механізм самовідчуття, де запит, ключ та значення отримуються з однієї послідовності  │
│  або даних.                                                                                                     │
│     • **Перехресна увага:** механізм перехресної уваги


Інформація про увагу

1. Введення
   • Механізм уваги є компонентом мережі нейронів.
   • Він фокусується на певних елементах або аспектах вхідної інформації.
   • Вважає та поєднує вхідні дані для виробництва вагової суми.

2. Задача, яку розв'язує механізм уваги
   • Проблема довгострокових залежностей у послідовній даних.
   • Поніння складних зв’язків між елементами вхідної інформації.
   • Ванішання градієнтів під час зворотньої передачі через час.

3. Шагова механізм
    1. Вихід (Питання, Ключ, Значення)
       • Механізм уваги приймає три вхідних дані – запит, ключ і значення вектори.
    2. Рахунок вагових коефіцієнтів
        • Він розрахований добуток між запит та ключ векторами для розрахунку вагових коефіцієнтів.
    3. Нормалізація вагових коефіцієнтів
        • Розраховані вагові коефіцієнти нормалізуються за допомогою функції softmax.
    4. Обрахунок контекстного вектору
        • Вагова сума значення векторів розрахується на основі нормалізованих вагових коефіцієнтів

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯